# Group Analysis

Full-cohort psychometric + physiology analysis across 10 participants.

**Reads:** `data/group_results/ (10-participant summaries)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT = '../data/group_results'

# load csv results
hrv_results_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv')
pupil_results_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv')
duration_results_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv')

# indexed lookup
hrv_idx = hrv_results_df.set_index('Participant')
pupil_idx = pupil_results_df.set_index('Participant')
dur_idx = duration_results_df.set_index('Participant')

for pid in hrv_idx.index:
    for s in [1, 2, 3]:
        col = f'Session {s:02d}'
        print(f"{pid}, Session {s} — HRV SDNN: {hrv_idx.loc[pid, col]:.2f}, Pupil STD: {pupil_idx.loc[pid, col]:.2f}, Duration STD: {dur_idx.loc[pid, col]:.2f}s")

# plot results
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

for ax, df, title, ylabel in zip(axes,
    [hrv_results_df, pupil_results_df, duration_results_df],
    ['Standard Deviation of HRV (SDNN) for Each Participant Across Three Sessions',
     'Standard Deviation of Pupil Dilation for Each Participant Across Three Sessions',
     'Standard Deviation of Psychometric Test Duration for Each Participant Across Three Sessions'],
    ['SDNN (ms)', 'Pupil Dilation STD', 'Duration STD (seconds)']):

    df.set_index('Participant').plot(ax=ax, marker='o')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore', category=FutureWarning, module='seaborn')

OUTPUT = '../data/group_results'

# load duration data
duration_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv')

# pivot by session
std_durations = duration_df.set_index('Participant')
std_durations.columns = ['Session 01', 'Session 02', 'Session 03']

print("Standard deviations (seconds):")
print(std_durations)

# transpose for plotting
std_durations = std_durations.T

fig, ax = plt.subplots(figsize=(15, 8))
sns.lineplot(data=std_durations, markers=True, dashes=False, ax=ax)

# reference lines
ax.axhline(y=300, color='blue', linestyle=':', label='Normal')
ax.axhline(y=600, color='purple', linestyle=':', label='Moderate Duration')
ax.axhline(y=900, color='green', linestyle=':', label='High Duration')

# merge legend handles
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, labels, title='Participant / Duration Levels')

plt.title('Standard Deviation of Answer Duration for Each Participant Across Three Sessions')
plt.ylabel('Standard Deviation of Answer Duration (seconds)')
plt.xlabel('Session')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
plt.close()

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load data
hrv_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv').set_index('Participant')
pupil_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv').set_index('Participant')
duration_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv').set_index('Participant')

metrics = {
    'HRV SDNN (ms)': hrv_df,
    'Pupil Dilation STD': pupil_df,
    'Duration STD (s)': duration_df
}

# --- normality tests ---
print("=== Shapiro-Wilk Normality Tests ===\n")
for name, df in metrics.items():
    for col in df.columns:
        stat, p = stats.shapiro(df[col].dropna())
        normal = "normal" if p > 0.05 else "non-normal"
        print(f"{name} {col}: W={stat:.3f}, p={p:.3f} ({normal})")
    print()

# --- repeated measures (Friedman test) ---
print("=== Friedman Test (repeated measures) ===\n")
for name, df in metrics.items():
    s1, s2, s3 = df.iloc[:, 0], df.iloc[:, 1], df.iloc[:, 2]
    stat, p = stats.friedmanchisquare(s1, s2, s3)
    sig = "*" if p < 0.05 else "ns"
    print(f"{name}: chi2={stat:.3f}, p={p:.3f} {sig}")
print()

# --- pairwise Wilcoxon tests ---
print("=== Pairwise Wilcoxon Signed-Rank Tests ===\n")
from itertools import combinations
pairs = list(combinations(range(3), 2))

for name, df in metrics.items():
    print(f"--- {name} ---")
    p_values = []
    for i, j in pairs:
        stat, p = stats.wilcoxon(df.iloc[:, i], df.iloc[:, j])
        p_values.append(p)

    # Holm-Bonferroni correction
    sorted_idx = np.argsort(p_values)
    corrected = np.zeros(len(p_values))
    m = len(p_values)
    for rank, idx in enumerate(sorted_idx):
        corrected[idx] = min(p_values[idx] * (m - rank), 1.0)

    for k, (i, j) in enumerate(pairs):
        sig = "*" if corrected[k] < 0.05 else "ns"
        print(f"  {df.columns[i]} vs {df.columns[j]}: p={p_values[k]:.3f}, p_corrected={corrected[k]:.3f} {sig}")
    print()

# --- Cohen's d effect sizes ---
print("=== Cohen's d (Session pairs) ===\n")
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx-1)*x.std()**2 + (ny-1)*y.std()**2) / (nx+ny-2))
    return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else 0.0

for name, df in metrics.items():
    print(f"--- {name} ---")
    for i, j in pairs:
        d = cohens_d(df.iloc[:, i], df.iloc[:, j])
        size = "large" if abs(d) >= 0.8 else "medium" if abs(d) >= 0.5 else "small"
        print(f"  {df.columns[i]} vs {df.columns[j]}: d={d:.3f} ({size})")
    print()

# --- 95% confidence intervals ---
print("=== 95% Confidence Intervals ===\n")
for name, df in metrics.items():
    print(f"--- {name} ---")
    for col in df.columns:
        data = df[col].dropna()
        n = len(data)
        mean = data.mean()
        se = data.std() / np.sqrt(n)
        ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=se)
        print(f"  {col}: mean={mean:.3f}, 95% CI=[{ci[0]:.3f}, {ci[1]:.3f}]")
    print()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load data
hrv_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv')
pupil_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv')
duration_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv')

# melt to long format
def melt_df(df, value_name):
    return df.melt(id_vars='Participant', var_name='Session', value_name=value_name)

hrv_long = melt_df(hrv_df, 'HRV SDNN (ms)')
pupil_long = melt_df(pupil_df, 'Pupil Dilation STD')
duration_long = melt_df(duration_df, 'Duration STD (s)')

# violin plots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, data, col, title in zip(axes,
    [hrv_long, pupil_long, duration_long],
    ['HRV SDNN (ms)', 'Pupil Dilation STD', 'Duration STD (s)'],
    ['HRV SDNN', 'Pupil Dilation STD', 'Duration STD']):

    sns.violinplot(data=data, x='Session', y=col, ax=ax, inner='box', palette='Set2')
    sns.stripplot(data=data, x='Session', y=col, ax=ax, color='black', alpha=0.5, size=4)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('Distribution of Biometric Metrics Across Sessions', y=1.02)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

OUTPUT = '../data/group_results'

# load data
hrv_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv').set_index('Participant')
pupil_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv').set_index('Participant')
duration_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv').set_index('Participant')

metrics = {
    'HRV SDNN': hrv_df,
    'Pupil STD': pupil_df,
    'Duration STD': duration_df
}

# --- Jonckheere-Terpstra trend test ---
def jonckheere_test(groups):
    k = len(groups)
    J = 0
    for i in range(k):
        for j in range(i+1, k):
            for xi in groups[i]:
                for xj in groups[j]:
                    if xj > xi:
                        J += 1
                    elif xj == xi:
                        J += 0.5
    
    # expected and variance
    ns = [len(g) for g in groups]
    N = sum(ns)
    E_J = (N**2 - sum(n**2 for n in ns)) / 4
    var_num = N**2 * (2*N + 3) - sum(n**2 * (2*n + 3) for n in ns)
    var_J = var_num / 72
    z = (J - E_J) / np.sqrt(var_J) if var_J > 0 else 0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return J, z, p

print("=== Habituation Analysis (Trend Tests) ===\n")
print("Tests whether biometric variability decreases across sessions\n")

for name, df in metrics.items():
    groups = [df.iloc[:, i].dropna().values for i in range(3)]
    J, z, p = jonckheere_test(groups)
    
    means = [g.mean() for g in groups]
    direction = "decreasing" if means[2] < means[0] else "increasing"
    sig = "*" if p < 0.05 else "ns"
    
    print(f"{name}: J={J:.0f}, z={z:.2f}, p={p:.3f} {sig}")
    print(f"  Session means: {[f'{m:.2f}' for m in means]} ({direction})")
    print()

# --- participant profiling ---
print("=== Participant Stress Profiles ===\n")

profiles = []
for pid in hrv_df.index:
    hrv_vals = hrv_df.loc[pid].values
    pupil_vals = pupil_df.loc[pid].values
    dur_vals = duration_df.loc[pid].values
    
    # trend direction
    hrv_slope = np.polyfit(range(3), hrv_vals, 1)[0]
    pupil_slope = np.polyfit(range(3), pupil_vals, 1)[0]
    dur_slope = np.polyfit(range(3), dur_vals, 1)[0]
    
    # classify pattern
    if hrv_slope < 0 and pupil_slope < 0:
        pattern = "habituating"
    elif hrv_slope > 0 and pupil_slope > 0:
        pattern = "sensitizing"
    else:
        pattern = "mixed"
    
    # overall variability
    overall = np.mean([np.std(hrv_vals), np.std(pupil_vals), np.std(dur_vals)])
    
    profiles.append({
        'Participant': pid,
        'HRV slope': hrv_slope,
        'Pupil slope': pupil_slope,
        'Duration slope': dur_slope,
        'Pattern': pattern,
        'Variability': overall
    })

prof_df = pd.DataFrame(profiles)
print(prof_df.to_string(index=False, float_format='%.3f'))

# pattern counts
print(f"\nPatterns: {prof_df['Pattern'].value_counts().to_dict()}")

# --- trajectory plots ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sessions = ['S01', 'S02', 'S03']

for ax, (name, df) in zip(axes, metrics.items()):
    for pid in df.index:
        vals = df.loc[pid].values
        pattern = prof_df[prof_df['Participant'] == pid]['Pattern'].iloc[0]
        color = 'tab:blue' if pattern == 'habituating' else 'tab:red' if pattern == 'sensitizing' else 'tab:gray'
        alpha = 0.7 if pattern != 'mixed' else 0.3
        ax.plot(sessions, vals, 'o-', color=color, alpha=alpha, linewidth=1.5, label=pid)
    
    # group mean
    mean_vals = df.mean().values
    ax.plot(sessions, mean_vals, 's--', color='black', linewidth=2.5, markersize=10, label='Group Mean', zorder=10)
    
    ax.set_title(name)
    ax.set_ylabel(name)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

# legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='tab:blue', label='Habituating'),
    Line2D([0], [0], color='tab:red', label='Sensitizing'),
    Line2D([0], [0], color='tab:gray', label='Mixed'),
    Line2D([0], [0], color='black', linestyle='--', marker='s', label='Group Mean')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05))
plt.suptitle('Individual Trajectories Across Sessions (colored by adaptation pattern)')
plt.tight_layout()
plt.show()
plt.close()

# --- session change heatmap ---
fig, ax = plt.subplots(figsize=(10, 8))
change_data = pd.DataFrame(index=hrv_df.index)
for name, df in metrics.items():
    change_data[f'{name}\nS1→S2'] = ((df.iloc[:, 1] - df.iloc[:, 0]) / df.iloc[:, 0] * 100)
    change_data[f'{name}\nS2→S3'] = ((df.iloc[:, 2] - df.iloc[:, 1]) / df.iloc[:, 1] * 100)

import seaborn as sns
sns.heatmap(change_data, cmap='RdYlGn_r', center=0, annot=True, fmt='.0f',
            linewidths=0.5, cbar_kws={'label': '% Change'}, ax=ax)
ax.set_title('Session-to-Session % Change per Participant')
plt.tight_layout()
plt.show()
plt.close()